# Job Market Demand Forecasting — Global Deep Learning Model

**Objective:** Forecast 6-month global hiring demand per sector using a recurrent neural network trained on combined data from 6 countries (US, AU, CA, DE, FR, GB).

**Data:** Indeed Job Postings Index (Feb 2020 – Apr 2026). The index is normalized so 100 = the pre-COVID baseline (February 2020). The global index per sector is the mean across all 6 countries.

**Approach:** Combine all regional datasets into a single global index per sector, denoise with STL decomposition, engineer time-series features, then train an ensemble of LSTM / GRU / Bidirectional LSTM models and select the best architecture for the forecast.

## Step 1 — Load All Country Datasets

Load the six per-country CSV files from the Kaggle dataset directory. Each country sits in its own subfolder.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['axes.grid']  = True
plt.rcParams['grid.alpha'] = 0.3
INPUT_DIR = '/kaggle/input/datasets/kimminh21/job-postings'
print("Files in dataset:")
print("=" * 60)
for dirname, _, filenames in os.walk(INPUT_DIR):
    level = dirname.replace(INPUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(dirname)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in sorted(filenames):
        filepath = os.path.join(dirname, f)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{subindent}{f} ({size_mb:.2f} MB)")
COUNTRIES = {
    'US': 'job_postings_by_sector_US.csv',
    'AU': 'job_postings_by_sector_AU.csv',
    'CA': 'job_postings_by_sector_CA.csv',
    'DE': 'job_postings_by_sector_DE.csv',
    'FR': 'job_postings_by_sector_FR.csv',
    'GB': 'job_postings_by_sector_GB.csv',
}
def smart_read_csv(filepath):
    """Auto-detect separator (',' or ';') and parse the date column robustly."""
    with open(filepath, 'r', encoding='utf-8') as f:
        first_line = f.readline()
    sep = ';' if first_line.count(';') > first_line.count(',') else ','
    df = pd.read_csv(filepath, sep=sep)
    # Date column may be 'date' or capitalized; normalize
    date_col = next((c for c in df.columns if c.lower() == 'date'), None)
    if date_col is None:
        raise ValueError(f"No date column found in {filepath}. Columns: {list(df.columns)}")
    if date_col != 'date':
        df = df.rename(columns={date_col: 'date'})
    # Try both day-first and month-first parsing
    parsed = pd.to_datetime(df['date'], errors='coerce', dayfirst=False)
    if parsed.isna().any():
        parsed = pd.to_datetime(df['date'], errors='coerce', dayfirst=True)
    df['date'] = parsed
    return df
dfs_raw = {}
for country, filename in COUNTRIES.items():
    filepath = os.path.join(INPUT_DIR, country, filename)
    if os.path.exists(filepath):
        df_c = smart_read_csv(filepath)
        df_c['country'] = country
        dfs_raw[country] = df_c
        print(f"✓ {country}: {df_c.shape[0]:>7,} rows | "
              f"{df_c['display_name'].nunique():>2} sectors | "
              f"{df_c['date'].min().date()} → {df_c['date'].max().date()}")
    else:
        print(f"✗ {country}: file not found at {filepath}")
print(f"\n✓ Loaded {len(dfs_raw)} countries")

## Step 2 — Aggregate to Monthly Global Index

We keep only `new postings` (fresh hiring) and the chosen sectors. After computing a per-country monthly mean for each sector, the **global index** is calculated as the mean across all countries — one combined time series per sector.

In [ ]:
DESIRED_SECTORS = [
    'Software Development',
    'Data & Analytics',
    'IT Systems & Solutions',
    'Project Management',
    'Marketing',
    'Management',
    'Banking & Finance',
    'Human Resources',
]

# Keep only sectors that exist in every country
sector_sets = [set(dfs_raw[c]['display_name'].unique()) for c in COUNTRIES]
common_all  = set.intersection(*sector_sets)
SECTORS = [s for s in DESIRED_SECTORS if s in common_all]
dropped = [s for s in DESIRED_SECTORS if s not in common_all]
print(f"Sectors kept: {len(SECTORS)} / {len(DESIRED_SECTORS)}")
for s in SECTORS:
    print(f"  ✓ {s}")
if dropped:
    print("Dropped (not available in every country):")
    for s in dropped:
        print(f"  ✗ {s}")

# Per-country monthly aggregation
country_monthly = {}
for c in COUNTRIES:
    df = dfs_raw[c]
    df = df[(df['variable'] == 'new postings') & (df['display_name'].isin(SECTORS))].copy()
    df['period'] = df['date'].dt.to_period('M')
    m = (df.groupby(['period', 'display_name'])['indeed_job_postings_index']
           .mean().unstack().sort_index())
    m.index = m.index.to_timestamp()
    country_monthly[c] = m[SECTORS]

# Global = mean across all 6 countries
global_monthly = sum(country_monthly[c] for c in COUNTRIES) / len(COUNTRIES)
print(f"\nGlobal monthly index: {global_monthly.shape[0]} months × {global_monthly.shape[1]} sectors")
print(f"Date range: {global_monthly.index.min().date()} → {global_monthly.index.max().date()}")
print(f"Index range: {global_monthly.values.min():.1f} — {global_monthly.values.max():.1f}")

## Step 3 — STL Decomposition

STL separates each sector's global series into trend + seasonal + residual. We drop the residual (random noise) and keep trend + seasonal — a cleaner signal to learn from. The `robust=True` flag down-weights the COVID crash months so they don't distort the decomposition.

In [ ]:
from statsmodels.tsa.seasonal import STL

g_trend = pd.DataFrame(index=global_monthly.index, columns=global_monthly.columns, dtype=float)
g_seas  = pd.DataFrame(index=global_monthly.index, columns=global_monthly.columns, dtype=float)

for sector in global_monthly.columns:
    result = STL(global_monthly[sector], period=12, robust=True).fit()
    g_trend[sector] = result.trend
    g_seas[sector]  = result.seasonal

global_denoised = g_trend + g_seas
print(f"Denoised global index: {global_denoised.shape}")

## Step 4 — Visualization

A heatmap of the global index for all selected sectors, plus a per-sector trend overlay so we can see each sector's trajectory clearly.

In [ ]:
categories = global_monthly.columns.tolist()
n_sectors  = len(categories)
colors     = [plt.cm.tab10(i % 10) for i in range(n_sectors)]

# Heatmap
fig, ax = plt.subplots(figsize=(16, max(4, n_sectors * 0.4)))
im = ax.imshow(global_monthly.T.values, aspect='auto', cmap='RdYlGn',
               vmin=50, vmax=200, interpolation='nearest')
ax.set_yticks(range(n_sectors)); ax.set_yticklabels(categories, fontsize=10)
xtick_pos = range(0, len(global_monthly), 6)
ax.set_xticks(list(xtick_pos))
ax.set_xticklabels([global_monthly.index[i].strftime('%Y-%m') for i in xtick_pos],
                   rotation=45, fontsize=9)
plt.colorbar(im, ax=ax, label='Global Indeed Index (100 = Feb 2020 baseline)')
ax.set_title('Global Job Postings Index — All Sectors (averaged across 6 countries)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Per-sector line plot
n_cols = 2
n_rows = (n_sectors + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows), sharex=True)
axes = axes.flatten() if n_sectors > 1 else [axes]
for i, sec in enumerate(categories):
    ax = axes[i]
    ax.plot(global_monthly.index, global_monthly[sec], lw=2, color=colors[i])
    ax.fill_between(global_monthly.index, global_monthly[sec], 100, alpha=0.15,
                    color='green' if global_monthly[sec].mean() > 100 else 'red')
    ax.axhline(100, color='gray', ls='--', lw=1)
    ax.set_title(sec, fontsize=10, fontweight='bold')
    ax.set_ylabel('Index')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
for j in range(n_sectors, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Global Index by Sector', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5 — Feature Engineering & Preprocessing

We transform the denoised global series into model-ready features: first differencing for stationarity, sin/cos month encoding for seasonality, a 3-month rolling standard deviation for volatility, and a binary COVID flag for March–June 2020. Everything is scaled to [-1, 1] using train-set statistics only, then converted into 12-month sliding windows.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

LOOK_BACK = 12

values     = global_denoised.values.astype(np.float32)
raw_values = global_monthly.values.astype(np.float32)

diff_values = np.diff(values, axis=0)
diff_index  = global_monthly.index[1:]

months_arr = diff_index.month.values
month_sin  = np.sin(2 * np.pi * months_arr / 12).reshape(-1, 1)
month_cos  = np.cos(2 * np.pi * months_arr / 12).reshape(-1, 1)

rolling_std = (pd.DataFrame(values, index=global_monthly.index)
                 .rolling(3, min_periods=1).std().values[1:])

covid_flag = np.asarray((diff_index >= '2020-03') & (diff_index <= '2020-06'),
                        dtype=float).reshape(-1, 1)

extra = np.hstack([month_sin, month_cos, rolling_std, covid_flag])

N_SECTORS      = diff_values.shape[1]
N_FEATURES_IN  = N_SECTORS + extra.shape[1]
N_FEATURES_OUT = N_SECTORS

print(f"Sectors                : {N_SECTORS}")
print(f"Extra features         : {extra.shape[1]} (2 seasonal + {rolling_std.shape[1]} volatility + 1 covid)")
print(f"Total input features   : {N_FEATURES_IN}")
print(f"Output targets         : {N_FEATURES_OUT}")

# Chronological split
n_diff    = len(diff_values)
train_end = int(n_diff * 0.70)
val_end   = int(n_diff * 0.85)

diff_train,  diff_val,  diff_test  = diff_values[:train_end], diff_values[train_end:val_end], diff_values[val_end:]
extra_train, extra_val, extra_test = extra[:train_end],       extra[train_end:val_end],       extra[val_end:]

scaler_diff  = MinMaxScaler(feature_range=(-1, 1))
scaler_extra = MinMaxScaler()

diff_train_s = scaler_diff.fit_transform(diff_train)
diff_val_s   = scaler_diff.transform(diff_val)
diff_test_s  = scaler_diff.transform(diff_test)

extra_train_s = scaler_extra.fit_transform(extra_train)
extra_val_s   = scaler_extra.transform(extra_val)
extra_test_s  = scaler_extra.transform(extra_test)

train_feat = np.hstack([diff_train_s, extra_train_s])
val_feat   = np.hstack([diff_val_s,   extra_val_s])
test_feat  = np.hstack([diff_test_s,  extra_test_s])

def create_sequences(feat, target, look_back):
    X, y = [], []
    for i in range(len(feat) - look_back):
        X.append(feat[i:i+look_back])
        y.append(target[i+look_back])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_feat, diff_train_s, LOOK_BACK)
X_val,   y_val   = create_sequences(
    np.vstack([train_feat[-LOOK_BACK:], val_feat]),
    np.vstack([diff_train_s[-LOOK_BACK:], diff_val_s]), LOOK_BACK)
X_test,  y_test  = create_sequences(
    np.vstack([val_feat[-LOOK_BACK:], test_feat]),
    np.vstack([diff_val_s[-LOOK_BACK:], diff_test_s]), LOOK_BACK)

print(f"\nX_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}")

## Step 6 — Define Three Recurrent Architectures

We compare LSTM, GRU, and Bidirectional LSTM with matched parameter counts. All three use identical regularization (L2 + dropout) and the Huber loss, which is robust to outlier months.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

def build_model(seed, model_type='lstm'):
    tf.keras.utils.set_random_seed(seed)
    if model_type == 'lstm':
        rnn = LSTM(48, input_shape=(LOOK_BACK, N_FEATURES_IN),
                   dropout=0.20, recurrent_dropout=0.20,
                   kernel_regularizer=l2(5e-3), recurrent_regularizer=l2(5e-3))
    elif model_type == 'gru':
        rnn = GRU(48, input_shape=(LOOK_BACK, N_FEATURES_IN),
                  dropout=0.20, recurrent_dropout=0.20,
                  kernel_regularizer=l2(5e-3), recurrent_regularizer=l2(5e-3))
    elif model_type == 'bilstm':
        rnn = Bidirectional(
            LSTM(24, dropout=0.20, recurrent_dropout=0.20,
                 kernel_regularizer=l2(5e-3), recurrent_regularizer=l2(5e-3)),
            input_shape=(LOOK_BACK, N_FEATURES_IN))
    model = Sequential([rnn, Dropout(0.35), Dense(N_FEATURES_OUT)],
                       name=f'{model_type}_seed{seed}')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.Huber(delta=1.0), metrics=['mae'])
    return model

print(f"{'Architecture':<14} {'Parameters':>12}")
print('-' * 28)
for mt in ['lstm', 'gru', 'bilstm']:
    m = build_model(0, mt)
    print(f"{mt.upper():<14} {m.count_params():>12,}")
    del m

## Step 7 — Train All Three Ensembles

Each architecture is trained as a 7-seed ensemble (21 models total). EarlyStopping halts training when validation loss plateaus; ReduceLROnPlateau halves the learning rate when progress stalls.

In [ ]:
MODEL_TYPES = ['lstm', 'gru', 'bilstm']
N_ENSEMBLE  = 7
SEEDS       = [7, 23, 42, 101, 314, 1729, 2718]

all_ensembles  = {}
all_histories  = {}

for mt in MODEL_TYPES:
    print(f"\n{'─'*60}")
    print(f"  Training {mt.upper()} ensemble")
    print(f"{'─'*60}")
    models, histories = [], []
    for seed in SEEDS:
        mdl = build_model(seed=seed, model_type=mt)
        cbs = [EarlyStopping(monitor='val_loss', patience=20,
                             restore_best_weights=True, verbose=0),
               ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                 patience=8, min_lr=1e-5, verbose=0)]
        h = mdl.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=250, batch_size=8, callbacks=cbs, verbose=0)
        models.append(mdl); histories.append(h)
        print(f"  seed {seed:>4} | best val_loss = {min(h.history['val_loss']):.4f}  ({len(h.history['loss'])} epochs)")
    all_ensembles[mt] = models
    all_histories[mt] = histories

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, mt in zip(axes, MODEL_TYPES):
    for k, h in enumerate(all_histories[mt]):
        ax.plot(h.history['val_loss'], alpha=0.6, label=f'seed {SEEDS[k]}')
    ax.set_title(f'{mt.upper()} — Validation Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(fontsize=6)
plt.tight_layout()
plt.show()

def ensemble_predict(models, X, scaler):
    preds_s = np.stack([m.predict(X, verbose=0) for m in models], axis=0)
    mean_d  = scaler.inverse_transform(preds_s.mean(axis=0))
    mem_d   = np.stack([scaler.inverse_transform(p) for p in preds_s], axis=0)
    return mean_d, mem_d

## Step 8 — Evaluation: RMSE, MAE, R²

We evaluate all three ensembles on the held-out test set and compare against naive (last-value) and seasonal-naive (same-month-last-year) baselines.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_predictions = {}
for mt in MODEL_TYPES:
    mean_diff, _ = ensemble_predict(all_ensembles[mt], X_test, scaler_diff)
    anchor = values[val_end : val_end + len(mean_diff)]
    model_predictions[mt] = anchor + mean_diff

y_true     = raw_values[val_end+1 : val_end+1+len(model_predictions['lstm'])]
test_dates = global_monthly.index[val_end+1 : val_end+1+len(y_true)]

y_naive    = np.vstack([raw_values[val_end].reshape(1,-1), y_true[:-1]])
y_seasonal = np.vstack([raw_values[global_monthly.index.get_loc(d - pd.DateOffset(years=1))]
                        for d in test_dates])

def compute_metrics(yt, yp):
    return {'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAE':  mean_absolute_error(yt, yp),
            'R2':   r2_score(yt, yp)}

yt_flat = y_true.flatten()
comparison = {
    'Naive (last value)' : compute_metrics(yt_flat, y_naive.flatten()),
    'Seasonal-Naive'     : compute_metrics(yt_flat, y_seasonal.flatten()),
}
for mt in MODEL_TYPES:
    comparison[f'{mt.upper()} Ensemble'] = compute_metrics(yt_flat, model_predictions[mt].flatten())

print(f"{'Model':<25} {'RMSE':>8}  {'MAE':>8}  {'R²':>8}")
print("=" * 55)
for name, m in comparison.items():
    print(f"{name:<25} {m['RMSE']:>8.2f}  {m['MAE']:>8.2f}  {m['R2']:>8.4f}")

# Bar charts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
names = list(comparison.keys())
bar_c = ['#9ca3af', '#9ca3af', '#2563eb', '#059669', '#dc2626']
for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    vs = [comparison[n][metric] for n in names]
    bars = ax.bar(range(len(names)), vs, color=bar_c, edgecolor='black', lw=0.5)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([n.replace(' Ensemble','').replace(' (last value)','') for n in names],
                       rotation=30, ha='right', fontsize=9)
    label = 'R²' if metric == 'R2' else metric
    ax.set_title(label, fontsize=12, fontweight='bold')
    for bar, v in zip(bars, vs):
        fmt = f'{v:.4f}' if metric == 'R2' else f'{v:.2f}'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                fmt, ha='center', va='bottom', fontsize=8, fontweight='bold')
fig.suptitle('Global Model — RMSE, MAE, R²', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Bias correction → pick best architecture
best_mt = None; best_rmse = float('inf')
bias_d  = {}; corr_d = {}
for mt in MODEL_TYPES:
    vd, _ = ensemble_predict(all_ensembles[mt], X_val, scaler_diff)
    va    = values[train_end:train_end+len(vd)]
    vt    = raw_values[train_end+1:train_end+1+len(vd)]
    b     = np.mean(vt - (va + vd), axis=0)
    yp    = model_predictions[mt]
    yp_c  = yp + b
    r_o   = np.sqrt(mean_squared_error(yt_flat, yp.flatten()))
    r_c   = np.sqrt(mean_squared_error(yt_flat, yp_c.flatten()))
    if r_c < r_o:
        bias_d[mt] = b; corr_d[mt] = yp_c; fr = r_c
        print(f"  {mt.upper()}: bias correction helps ({r_o:.2f} → {r_c:.2f})")
    else:
        bias_d[mt] = np.zeros_like(b); corr_d[mt] = yp; fr = r_o
        print(f"  {mt.upper()}: bias correction skipped ({r_o:.2f} → {r_c:.2f})")
    if fr < best_rmse:
        best_rmse = fr; best_mt = mt

print(f"\nBest architecture: {best_mt.upper()} (RMSE = {best_rmse:.2f})")

# Actual vs Predicted for each sector
mc = {'lstm': '#2563eb', 'gru': '#059669', 'bilstm': '#dc2626'}
n_cols = 2
n_rows = (n_sectors + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.5 * n_rows))
axes = axes.flatten() if n_sectors > 1 else [axes]
for i, sec in enumerate(categories):
    ax = axes[i]
    ax.plot(test_dates, y_true[:,i], 'k-o', lw=2.5, ms=5, label='Actual', zorder=5)
    for mt in MODEL_TYPES:
        yp = corr_d[mt][:,i]
        r  = np.sqrt(mean_squared_error(y_true[:,i], yp))
        r2 = r2_score(y_true[:,i], yp)
        ax.plot(test_dates, yp, color=mc[mt], lw=1.8, ls='--', marker='x', ms=4, alpha=0.85,
                label=f'{mt.upper()} ({r:.1f}, R²={r2:.2f})')
    ax.set_title(sec, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=8)
for j in range(n_sectors, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Actual vs Predicted — All Three Architectures', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 9 — 6-Month Global Forecast

Using the best architecture, we generate a 6-month recursive forecast. Each ensemble member produces an independent trajectory; the mean is the point forecast and the 10th–90th percentile spread is the uncertainty band.

In [ ]:
FORECAST_MONTHS = 6

full_diff_s  = scaler_diff.transform(diff_values)
full_extra_s = scaler_extra.transform(extra)
full_feat    = np.hstack([full_diff_s, full_extra_s])

def recursive_forecast(model, n_steps):
    last_window    = full_feat[-LOOK_BACK:].copy()
    last_abs       = values[-1].copy()
    last_month_num = global_monthly.index[-1].month
    recent_levels  = list(values[-3:])

    out = []
    for _ in range(n_steps):
        pred_diff_s   = model.predict(last_window[np.newaxis], verbose=0)[0]
        pred_diff_val = scaler_diff.inverse_transform(pred_diff_s.reshape(1, -1))[0]
        next_abs      = last_abs + pred_diff_val
        out.append(next_abs)

        recent_levels  = recent_levels[-2:] + [next_abs]
        next_rstd      = np.std(np.stack(recent_levels), axis=0, ddof=1)
        next_month_num = (last_month_num % 12) + 1
        next_sin       = np.sin(2 * np.pi * next_month_num / 12)
        next_cos       = np.cos(2 * np.pi * next_month_num / 12)

        next_extra_raw = np.hstack([[[next_sin, next_cos]], next_rstd.reshape(1, -1), [[0.0]]])
        next_extra_s   = scaler_extra.transform(next_extra_raw)
        next_row       = np.hstack([pred_diff_s.reshape(1, -1), next_extra_s]).squeeze()
        last_window    = np.vstack([last_window[1:], next_row])
        last_abs       = next_abs
        last_month_num = next_month_num
    return np.array(out)

best_models = all_ensembles[best_mt]
best_bias   = bias_d[best_mt]

member_fc = np.stack([recursive_forecast(m, FORECAST_MONTHS) for m in best_models])
fc_mean   = member_fc.mean(axis=0)
fc_low    = np.quantile(member_fc, 0.10, axis=0)
fc_high   = np.quantile(member_fc, 0.90, axis=0)
fc_corr   = fc_mean + best_bias

fc_dates = pd.date_range(start=global_monthly.index[-1] + pd.DateOffset(months=1),
                         periods=FORECAST_MONTHS, freq='MS')

forecast_df = pd.DataFrame(fc_corr.round(1), index=fc_dates.strftime('%Y-%m'), columns=categories)
print(f"\n6-Month Global Forecast ({best_mt.upper()} Ensemble):")
print(forecast_df.to_string())

# Heatmap (historical + forecast)
fig, ax = plt.subplots(figsize=(14, max(4, n_sectors * 0.5)))
comb = np.hstack([raw_values[-12:].T, fc_corr.T])
clbl = ([d.strftime('%Y-%m') for d in global_monthly.index[-12:]]
        + fc_dates.strftime('%Y-%m').tolist())
im = ax.imshow(comb, aspect='auto', cmap='RdYlGn', vmin=50, vmax=200)
ax.set_yticks(range(n_sectors)); ax.set_yticklabels(categories, fontsize=10)
ax.set_xticks(range(len(clbl))); ax.set_xticklabels(clbl, rotation=45, fontsize=8)
ax.axvline(x=11.5, color='black', lw=2, ls='--')
ax.text(9.5, -1.0, 'Historical', fontsize=10)
ax.text(12.5, -1.0, '6-Month Forecast', fontsize=10, color='blue')
plt.colorbar(im, ax=ax, label='Global Index')
ax.set_title(f'Global Forecast — {best_mt.upper()} Ensemble', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Per-sector trajectory
HIST_MONTHS = 24
hist_dates  = global_monthly.index[-HIST_MONTHS:]

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten() if n_sectors > 1 else [axes]
summary_rows = []
for i, sec in enumerate(categories):
    ax = axes[i]
    hy  = raw_values[-HIST_MONTHS:, i]
    fy  = fc_corr[:, i]
    ly  = fc_low[:, i] + best_bias[i]
    hy2 = fc_high[:, i] + best_bias[i]
    cur = float(hy[-1]); fin = float(fy[-1])
    pct = (fin - cur) / cur * 100
    dc  = '#16a34a' if pct >= 0 else '#dc2626'

    ax.plot(hist_dates, hy, color=colors[i], lw=2.2, marker='o', ms=4, label='Historical (24m)')
    ax.plot([hist_dates[-1], fc_dates[0]], [hy[-1], fy[0]], 'k--', lw=1.2, alpha=0.6)
    ax.plot(fc_dates, fy, 'k--D', lw=2.4, ms=6, label='Forecast (6m)')
    ax.fill_between(fc_dates, ly, hy2, color=dc, alpha=0.18, label='Ensemble 10-90%')
    ax.axhline(100, color='gray', ls=':', lw=1, alpha=0.7)

    d = '▲' if pct >= 0 else '▼'
    ax.set_title(f'{sec}\n{d} {pct:+.1f}% over 6 months', fontsize=11, fontweight='bold', color=dc)
    ax.set_ylabel('Global Index')
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, fontsize=8)

    summary_rows.append({'Sector': sec, 'Current': round(cur, 1),
                         'Forecast (6m)': round(fin, 1),
                         'Change (pts)': round(fin - cur, 1),
                         'Change (%)': round(pct, 1)})

for j in range(n_sectors, len(axes)):
    axes[j].set_visible(False)
fig.suptitle(f'6-Month Global Trajectory — {best_mt.upper()} Ensemble',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

trajectory_df = pd.DataFrame(summary_rows).sort_values('Change (%)', ascending=False).reset_index(drop=True)
print("\nGlobal trajectory summary (ranked by projected change):")
print(trajectory_df.to_string(index=False))

# Ranked bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bar_df = trajectory_df.sort_values('Change (%)')
bar_color = ['#16a34a' if v >= 0 else '#dc2626' for v in bar_df['Change (%)']]
bars = ax.barh(bar_df['Sector'], bar_df['Change (%)'], color=bar_color, edgecolor='black', lw=0.6)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Projected change over 6 months (%)')
ax.set_title('Global Forecast Ranking — Where is hiring heading?',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, bar_df['Change (%)']):
    offset = 0.4 if val >= 0 else -0.4
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}%', va='center', ha=ha, fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusion

This notebook trained a recurrent neural network ensemble (LSTM / GRU / Bidirectional LSTM, 7 seeds each) to forecast global hiring demand across selected sectors. Combining all six country datasets into a single global index reduces country-specific noise and produces a stable signal of worldwide hiring trends.

The model is evaluated with RMSE, MAE, and R² against naive baselines, the best architecture is selected automatically based on validation-corrected test performance, and the 6-month forecast includes ensemble uncertainty bands derived from the spread across the 7 seeds.